In [ ]:
from pyspark.sql.session import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("intro_delta")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
        "io.delta:delta-spark_2.12:3.3.0,"
        "org.scala-lang:scala-library:2.12.18",
    )
    # Configurações essenciais do Delta
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # # Configurações do S3/MinIO
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.driver.memory", "12g")
    .config("spark.executor.memory", "12g")
    .config("spark.executor.memoryOverhead", "4g")
    # Configurações de compatibilidade
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .config("spark.sql.warehouse.dir", "s3a://sparkdw/")
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
    .enableHiveSupport()
    .getOrCreate()
)
spark

In [ ]:
dataframe = spark.sql(""" 
                    select 
                      
                        row_number() over (order by businessentityid) as buyer_key, 
                        * 
                        from delta. `s3a://bronze/advworks/raw_dimbuyer`
                """)

In [ ]:
spark.sql("SHOW TABLES IN icebergtb.advworks_rdb ").show(truncate=False)
spark.sql(""" select * from icebergtb.advworks_rdb.raw_dimtransactioncode.manifests""").show(
    truncate=False
)

In [ ]:
df = spark.sql(
    """ select row_number() over (order by businessentityid) as buyer_key, * from icebergtb.advworks_rawdb.raw_dimbuyer"""
).show()

df.createOrReplaceTempView()

In [ ]:
buyer_temp = spark.sql(""" 
                        SELECT
                            row_number() over (order by businessentityid) as buyer_key,
                            businessentityid AS buyer_id,    firstname,
                            lastname,                        birthdate,
                            
                            CAST(DATE_DIFF(CURRENT_DATE, birthdate) / 365 AS INT) AS age,

                            CASE
                                WHEN maritalstatus = 'M' THEN 'Married'
                                ELSE 'Single'
                            END AS maritalstatus,

                            CASE
                                WHEN gender = 'F' THEN 'Female'
                                ELSE 'Male'
                            END AS gender,

                            hiredate,
                            
                            CAST(DATE_DIFF(CURRENT_DATE, hiredate) / 365 AS INT) AS years_in_company,
                            
                            departmentid,                    name AS department_name,
                            groupname AS department_manager, jobtitle,
                            payfrequency,                    rate,
                            salarychange_date,               startdate,

                            CASE
                                WHEN enddate IS NULL AND startdate = salarychange_date THEN COALESCE(enddate, '2999-12-31')
                                WHEN enddate IS NULL AND (DAY(startdate) - DAY(salarychange_date) = 1) THEN salarychange_date
                                WHEN enddate IS NULL AND MAX(salarychange_date) < MAX(startdate) THEN COALESCE(enddate, '2999-12-31')
                                ELSE enddate
                            END AS enddate,

                            salariedflag AS is_salaried,    currentflag AS is_current,
                            
                            extracted_date,
                            
                            COALESCE(modified_date, '2999-12-31') AS modified_date,
                            
                            eltversion
                            
                        FROM icebergtb.advworks_rawdb.raw_dimbuyer
                        
                        GROUP BY
                            businessentityid, firstname,            lastname,
                            birthdate,        maritalstatus,        gender,
                            hiredate,         departmentid,         name,
                            groupname,        jobtitle,             payfrequency,
                            rate,             salarychange_date,    startdate,
                            enddate,          salariedflag,         currentflag,
                            extracted_date,    modified_date,        eltversion
""").show()

In [ ]:
# spark.sql("""SELECT * FROM delta.`s3a://silver/advworks/dim_buyer/`""").show()

spark.read.format("delta").load("s3a://silver/advworks/dim_buyer")

In [ ]:
spark.sql("""show catalogs""").show()
spark.sql(""" show databases in spark_catalog """).show()
spark.sql(""" show tables in spark_catalog.default""").show()

In [ ]:
spark.sql(""" select * from spark_catalog.default.dim_product""").show()